# ISLP Chapter 5 Lab
#### Author: Thomas Fitzgerald
This notebook contains my work for the lab accompanying Chapter 5 of [*An Introduction to Statistical Learning with Python* ("*ISLP*")](https://www.statlearning.com/). The fifth chapter of *ISLP* focuses on the use of resamlpling methods for analysis of models (specifically cross-validation and the bootstrap). This lab makes use of libraries introduced in the previous lab (**numpy**, **statsmodel**, **ISLP**, and **sklearn**) as well as portions of the **functools** library. To begin, the imports from the previous labs are pulled in:

In [2]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (
    ModelSpec as MS,
    summarize,
    poly
)
from sklearn.model_selection import train_test_split

Next, new imports are pulled:

In [3]:
from functools import partial
from sklearn.model_selection import (
    cross_validate,
    KFold,
    ShuffleSplit
)
from sklearn.base import clone
from ISLP.models import sklearn_sm

### The Validation Set Approach
In this section, the validation set approach is used to estimate the test error rate of a set of models on the **Auto** data set. The **train_test_split()** function is used to split the data into validation sets. To begin, two sets are used. These sets are created with a random seed to ensure that the results are reproducable.

In [4]:
Auto = load_data('Auto')
n = len(Auto)
Auto_train, Auto_val = train_test_split(
    Auto,
    test_size = int(n / 2),
    random_state = 0
)

With the testing and training sets established, a linear regression model can be fit on the training data.

In [5]:
hp = MS(['horsepower'])
X_train = hp.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

Now, the results of the model can be tested on the validation (testing) data set. Additionally, the MSE of the model can be calculated.

In [6]:
X_val = hp.fit_transform(Auto_val)
y_val = Auto_val['mpg']
val_pred = results.predict(X_val)
np.mean((y_val - val_pred) ** 2)

np.float64(23.61661706966988)

As can be seen, the validation MSE is roughly 23.6 using linear regression on this validation set. The validation MSE can also be tested using a higher-order polynomial regression. To do so, a function **eval_MSE()** must be defined to take a model string and training and test sets and returns the MSE on the test set. 

In [7]:
def eval_MSE(terms, response, train, test):
    mn = MS(terms)
    X_train = mn.fit_transform(train)
    y_train = train[response]
    X_test = mn.transform(test)
    y_test = test[response]
    
    model = sm.OLS(y_train, X_train).fit()
    results = model.predict(X_test)
    return np.mean((y_test - results) ** 2)

This function will be used to estimate the validation MSE using linear, quadratic, and cubic fits. 

In [8]:
MSE = np.zeros(3)
for i, v in enumerate(range(1, 4)):
    MSE[i] = eval_MSE(
        [poly('horsepower', v)],
        'mpg',
        Auto_train,
        Auto_val
    )
MSE

array([23.61661707, 18.76303135, 18.79694163])

The resulting MSEs are roughly 23.62, 18.76, and 18.77 for linear, quadratic, and cubic approaches, respectively. Based on the results, it appears that the quadratic approach is most suitable (though cubic is very close). If a different training and testing split were used, the results could be expected to be a bit different.

In [11]:
Auto_train, Auto_val = train_test_split(
    Auto, 
    test_size=196, 
    random_state = 3
)
MSE = np.zeros(3)
for i, v in enumerate(range(1, 4)):
    MSE[i] = eval_MSE(
        [poly('horsepower', v)],
        'mpg',
        Auto_train,
        Auto_val
    )
MSE

array([20.75540796, 16.94510676, 16.97437833])

The above results are consistent with the first test of linear vs. quadratic vs. cubic models. The quadratic model is a significant improvement ove the linear model with little difference between quadratic and cubic models.

### Cross-Validation
In theory, cross-validation can be computed for any generalized-linear model. However, the simplest way to perform cross-validation in Python is to use the **sklearn** library, which has a different API than the **statsmodels** library that has been used to this point in fitting GLMs.

This is a common problem where a tool exists to solve one problem, but another tool is needed to solve a problem on top of that problem. Such instances require a wrapper function that allows the results of the first tool to be passed to the next. Luckily, the **ISLP** library includes the **sklearn_sm()** wrapper class that enables the easy use of cross-validation tools from **sklearn** with models fit by **statsmodels**.

The **sklearn_sm()** class takes the **statsmodels** model as its first argument and can take two additional arguments:
1. **model_str**: Which can be used to specify a formula.
2. **model_args**: Which should be a dictionary of additional arguments used when fitting the model.

For example, to specify a logistic regression model the **family** argument must be specified. To do so, pass in *model_args={'family':sm.families.Binomial()}*. Below, the wrapper is demonstrated/

In [16]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(
    hp_model,
    X, 
    Y,
    cv=Auto.shape[0]
)
cv_error = np.mean(cv_results['test_score'])
cv_error

np.float64(24.23151351792922)

The arguments to **cross_validate** are as follows:
1. The model to validate (must have **fit()**, **predict()**, and **score()** methods).
2. An array of features.
3. An array for the response.
4. **cv** an optional argument which takes an integer specifying the number folds to perform.

In the case above, the value of **cv** was set to the total number of observation. This results in *leave-one-out cross-validation* (LOOCV). 

The **cross_validate** function produces a dictionary with several components. The **test_score** component corresponds to the test MSE. This process can be repeated for increasingly complex polynomial fits, as was done with MSE in the previous section.

In [18]:
cv_errors = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    M_CV = cross_validate(
        M,
        X,
        Y,
        cv=Auto.shape[0]
    )
    cv_errors[i] = np.mean(M_CV['test_score'])
cv_errors

array([24.23151352, 19.24821312, 19.33498406, 19.42443031, 19.03321178])